# Deep Learning: Programming Assignment 2 — Rastreamento e Inferência
## Inferência com Rastreamento Recorrente (Trilha A: LSTM Motion Model)

**Requisito Oficial do PA2 (Entregável `inferencia.ipynb`):**
> *"Recebe o caminho de uma sequência qualquer, devolve o vídeo com as identidades coloridas de forma consistente e a contagem de objetos únicos. Roda sem retreinar."*

Este notebook carrega os pesos pré-treinados do modelo de movimento recorrente (`checkpoints/motion_lstm_best.pt`), executa o rastreador `RNNMotionTracker` com amortecimento de velocidade e inflação de incerteza (*rollout* sob oclusão), renderiza o vídeo animado com caixas delimitadoras coloridas de maneira consistente e contabiliza o total de identidades únicas.

In [1]:
%load_ext autoreload
%autoreload 2

import os
from IPython.display import Image as IPImage, display
from src.inference import run_tracking_inference

### 1. Parâmetros de Inferência

Defina o caminho da sequência desejada (`seq_path`).
- Se a pasta contiver `det/det.txt`, as detecções públicas são utilizadas.
- Se não houver `det.txt`, o pipeline executa o detector Faster R-CNN (COCO) com o `custom_nms` próprio.

In [2]:
# Caminho da sequência e pesos pré-treinados (SEM RETREINAR)
SEQ_PATH = "data/MOT17/train/MOT17-09-SDP"
MODEL_PATH = "checkpoints/motion_lstm_best.pt"
OUTPUT_VIDEO = "results/inferencia_output.gif"

# Executa o rastreador recorrente
res = run_tracking_inference(
    seq_path=SEQ_PATH,
    model_path=MODEL_PATH,
    output_video_path=OUTPUT_VIDEO,
    max_frames=40,
    fps=8,
    render_scale=0.5,
    velocity_damping=0.85,
    sigma_inflation=0.30,
    device="cpu"
)

Executando rastreamento recorrente em 40 quadros de MOT17-09-SDP...
Vídeo de inferência gerado com sucesso: results/inferencia_output.gif
RESUMO DA INFERÊNCIA
Sequência processada     : MOT17-09-SDP
Quadros analisados       : 40
Contagem de Objetos Únicos: 11 pedestres
Arquivo de saída         : results/inferencia_output.gif


### 2. Vídeo com Identidades Coloridas de Forma Consistente

Cada pedestre rastreado recebe uma cor única e estável (determinada via mapeamento de tom HSV com proporção áurea). O rastro de posição dos últimos 20 quadros e a tag do ID são plotados em tempo real sobre a imagem.

In [3]:
print(f"Total de pedestres únicos identificados ao longo do vídeo: {res['unique_objects_count']}")
print(f"IDs atribuídos: {res['unique_ids']}")

# Exibe o vídeo gerado diretamente no notebook
display(IPImage(filename=res["output_video_path"]))

Total de pedestres únicos identificados ao longo do vídeo: 11
IDs atribuídos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


### 💡 Resumo do Mecanismo de Identidade ao Longo do Tempo

1. **Associação Geométrica e Temporal:** Cada nova detecção é pareada contra a **caixa predita** pela rede recorrente LSTM para o quadro atual através do Algoritmo Húngaro sobre a matriz de IoU.
2. **Sob Oclusão:** Quando o detector não detecta um pedestre ativo, o rastreador entra em modo `lost` e executa um *rollout autoregressivo* com velocidade amortecida (`velocity_damping=0.85`), mantendo a track na posição esperada e sustentando a identidade.
3. **Ciclo de Vida:** Detecções sem correspondência geram novas tracks (`birth`). Tracks que permanecem mais de $k=15$ quadros consecutivas sem observação são extintas (`death`).